In [1]:
import tempfile
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import torch
from torchvision.transforms import v2
from torchvision.utils import save_image
from ultralytics import YOLO


In [ ]:
import json
import random

import numpy as np
from PIL import Image
from torchvision.transforms import v2 as T


def plot_results_from_paths(
    image_paths: list[Path],
    label_paths: list[Path],
    run_path: str,
    model_name: str = "",
    num_images: int = 6,
    seed: int | None = None, 
):
    pairs = list(zip(image_paths, label_paths))
    if seed is not None:
        random.seed(seed)
    sampled = random.sample(pairs, k=min(num_images, len(pairs)))

    transform = T.Compose([
        T.ToImage(),
        T.ToDtype(torch.float32, scale=True),
    ])

    model = YOLO(run_path).to(device="cpu")

    # 3) Prepare plotting grid
    nrows, ncols = int(num_images/3), 3
    fig, axes = plt.subplots(nrows, ncols, figsize=(22, 15), dpi=120)
    axes = axes.flatten()

    for idx, (img_path, lbl_path) in enumerate(sampled):
        if idx >= num_images:
            break

        ax = axes[idx]

        # Load & transform image 
        pil = Image.open(img_path)
        img_tensor = transform(pil)                # Tensor[c,w,h]
        img_np = img_tensor.squeeze().numpy()      # H×W array
        print(img_np.shape)
        c, h, w = img_np.shape
        img_np = np.transpose(img_np, (1,2,0))

        # Run YOLO (via a temp TIFF) 
        with tempfile.NamedTemporaryFile(suffix=".png") as tmp:
            save_image(img_tensor, tmp.name)       # writes a [1,H,W] TIFF
            results = model.predict(source=tmp.name, conf=0.3, verbose=False)
        pred_boxes = []
        for res in results:
            for box in res.boxes.xyxy.cpu().numpy():
                pred_boxes.append(box.tolist())    # [xmin,ymin,xmax,ymax]

        #  Load ground‑truth JSON 
        with open(lbl_path, 'r') as jf:
            data = json.load(jf)
        gt_boxes = data["bboxes"]                 # list of [min_row,min_col,max_row,max_col]

        #  Plot image & boxes 
        ax.imshow(img_np, origin="upper", extent=(0, w, h, 0),)
        ax.set_aspect("equal")

        # Ground truth in green
        for (xmin, ymin, xmax, ymax) in gt_boxes:
            rect = mpatches.Rectangle(
                (xmin, ymin), xmax - xmin, ymax-ymin,
                edgecolor="green",
                fill=False,
                linewidth=2,
                linestyle="--"
            )
            ax.add_patch(rect)

        # Predictions in red
        for (xmin, ymin, xmax, ymax) in pred_boxes:
            rect = mpatches.Rectangle(
                (xmin, ymin),
                xmax - xmin,
                ymax - ymin,
                edgecolor="red",
                fill=False,
                linewidth=1.5,
                linestyle="--"
            )
            ax.add_patch(rect)

        # Title with counts
        ax.text(
            0.02, 0.98,
            f"GT: {len(gt_boxes)}\nPred: {len(pred_boxes)}, {img_path}",
            transform=ax.transAxes,
            fontsize=14,
            verticalalignment='top',
            bbox=dict(facecolor='white', alpha=0.6, edgecolor='none')
        )

        ax.set_title(f"Sample {idx + 1}", fontsize=15)
        ax.axis('off')

    fig.suptitle(f"GT (green) vs. Pred({model_name}) (red)", fontsize=20)
    fig.subplots_adjust(wspace=0.05, hspace=0.05)
    plt.show()

In [ ]:
image_paths = sorted(Path("../data/carpk/images/val").glob("*.png"))
label_paths = [Path("../data/carpk/labels/xyxy") / f"{p.stem}.json" for p in image_paths]

plot_results_from_paths(
    image_paths=image_paths,
    label_paths=label_paths,
    run_path="../runs/detect/train190/weights/best.pt",
    model_name="YOLOv8n",
    num_images=9,
    seed=42
)

FileNotFoundError: [Errno 2] No such file or directory: '../runs/detect/train199/weights/best.pt'